# Ingestion

## Website network -> (Url,HTML) dictionary

In [ ]:
import requests
import time
import random
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/125.0.0.0 Safari/537.36"
)

def _crawl(
    current_url: str,
    depth: int,
    max_depth: int,
    root_domain: str,
    html_pages: dict,
    user_agent=USER_AGENT,
    min_delay=0.5,
    max_delay=1.5,
):
    if current_url in html_pages:
        return

    print(f"Crawling website: {current_url}...")
    
    # Add randomness to simulate human behaviour
    time.sleep(random.uniform(min_delay, max_delay))

    try:
        response = requests.get(
            current_url,
            timeout=10,
            headers={"User-Agent": user_agent},
        )
        response.raise_for_status()

        html = response.text
        html_pages[current_url] = html

        if depth == max_depth:
            return

        soup = BeautifulSoup(html, "html.parser")
        website_links = soup.find_all("a", href=True)

        for link in website_links:
            absolute_url = urljoin(current_url, link["href"])
            parsed_url = urlparse(absolute_url)

            # Stay within the same domain
            if parsed_url.netloc == root_domain:
                cleaned_url = parsed_url.geturl()


                _crawl(
                    current_url=cleaned_url,
                    depth=depth + 1,
                    max_depth=max_depth,
                    root_domain=root_domain,
                    html_pages=html_pages,
                )

    except Exception as e:
        print(f"Error crawling {current_url}: {e}")


def crawl_html(url: str, max_depth: int) -> list[str]:
    html_pages = {}
    root_domain = urlparse(url).netloc

    _crawl(
        current_url=url,
        depth=0,
        max_depth=max_depth,
        root_domain=root_domain,
        html_pages=html_pages,
    )

    return html_pages

In [9]:
INITIAL_CRAWLED_URL = "https://nau64.com/about/"
DEPTH = 0
pages = crawl_html(INITIAL_CRAWLED_URL, max_depth=DEPTH)

## (URL,HTML) dictionary -> CSV

In [7]:
import csv


def save_to_csv(
    html_by_url: dict[str, str],
    output_file: str,
):
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)

        writer.writerow(["url", "raw_text"])
        for url, html in html_by_url.items():
            writer.writerow([url, html])

In [10]:
CRAWLED_DATA_PATH = "crawled_data.csv"
save_to_csv(pages, CRAWLED_DATA_PATH)

## CSV -> Cleaned CSV

In [17]:
import csv
import re


def extract_body(html: str) -> str:
    match = re.search(r"<body[^>]*>(.*?)</body>", html, flags=re.DOTALL)

    if match:
        return match.group(1)

    return ""


def extract_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text()


def clean_html_csv(
    input_csv: str,
    output_csv: str,
):
    with open(input_csv, "r", encoding="utf-8", newline="") as infile, \
         open(output_csv, "w", encoding="utf-8", newline="") as outfile:

        reader = csv.DictReader(infile)
        fieldnames = list(reader.fieldnames) + ["cleaned_html"]

        writer = csv.DictWriter(
            outfile,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in reader:
            html = row["raw_text"]

            html_body = extract_body(html)
            body_text = extract_text(html_body)

            row["cleaned_html"] = body_text
            writer.writerow(row)

In [3]:
CLEANED_CRAWLED_DATA_PATH = "cleaned_crawled_data.csv"

In [ ]:
clean_html_csv(
    input_csv=CRAWLED_DATA_PATH,
    output_csv=CLEANED_CRAWLED_DATA_PATH,
)

## Cleaned CSV -> Chunks

In [ ]:
import csv


def chunk_text(text: str, chunk_size: int) -> list[str]:
    return [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
    ]


def create_chunked_csv(
    input_csv: str,
    output_csv: str,
    chunk_size: int = 1000,
):
    with open(input_csv, "r", encoding="utf-8", newline="") as infile:
        reader = csv.DictReader(infile)

        with open(output_csv, "w", encoding="utf-8", newline="") as outfile:
            writer = csv.DictWriter(
                outfile,
                fieldnames=["url", "chunk_id", "chunk_text"]
            )

            writer.writeheader()

            for row in reader:
                url = row["url"]
                text = row["cleaned_html"]

                chunks = chunk_text(text, chunk_size)

                for chunk_id, chunk in enumerate(chunks):
                    writer.writerow({
                        "url": url,
                        "chunk_id": chunk_id,
                        "chunk_text": chunk,
                    })

In [10]:
CHUNKED_DATA_PATH = "chunked_data.csv"

In [33]:
create_chunked_csv(
    input_csv=CLEANED_CRAWLED_DATA_PATH,
    output_csv=CHUNKED_DATA_PATH,
    chunk_size=500,
)

# Chunks -> Keyword Engine

In [34]:
import csv
from rank_bm25 import BM25Okapi


def tokenize(sentence: str):
    return sentence.lower().split()


def load_chunks(csv_file: str):
    chunks = []

    with open(csv_file, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            chunks.append({
                "url": row["url"],
                "chunk_id": row["chunk_id"],
                "chunk_text": row["chunk_text"],
            })

    return chunks


def build_bm25_index(chunks):
    tokenized_corpus = [
        tokenize(chunk["chunk_text"]) for chunk in chunks
    ]
    bm25 = BM25Okapi(tokenized_corpus)

    return bm25


def search(
    query: str,
    bm25: BM25Okapi,
    chunks,
    top_k: int = 5,
):
    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    ranked = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True,
    )

    return ranked[:top_k]

In [49]:
chunks = load_chunks(CHUNKED_DATA_PATH)
bm25 = build_bm25_index(chunks)
query = "Qué es nau64?"

results = search(
    query=query,
    bm25=bm25,
    chunks=chunks,
    top_k=3,
)

retrieved_documents = [doc["chunk_text"] for doc, _ in results]

for chunk, score in results:
    print(f"Score: {score:.4f}")
    print(f"URL: {chunk['url']}")
    print(chunk["chunk_text"])
    print("-" * 80)

Score: 4.6578
URL: https://nau64.com/about/


Skip to content







Menu


Home
Qué es nau64?
Clases de Ajedrez

Clases y Talleres
Clases para Niños
Clases Online


Torneos
Contacto
Nau64 en ChessCom
Uruguay
Área de Alumnos

Plataforma
Cursos y Materiales


 
Search for:






Search










Menu


nau64 en Chess.Com
Club nau64 en Chess.com
Súmate a Chess.Com
 


Twitch
YouTube
Instagram
Facebook
Twitter
TikTok
 







 
chess club
Academia de Ajedrez




Chatea con nosotros    
 










Menu


Home
Qué es nau64?
Clases de Ajedrez

--------------------------------------------------------------------------------
Score: 3.1203
URL: https://nau64.com/about/
ificación olímpica





Nicolás Kulik arrasa en el sabatino de la Academia





Frank y Federico brillan en el Rápido y Blitz Sabatino.





Open 135 el 6 de abril Torneo Pensado (FIDE)





La Batalla Infinita: Karpov vs. Kasparov





Sábados a Todo Ajedrez






lunes, junio 08, 2026














Qué es nau64? 

Un lugar don

# Prompts

In [50]:
expertise_area = "the chess club Nau64"

system_prompt = """
You are a chatbot, expert in {expertise_area}.

The inputs you will recieve are:
- A list of documents.
- A query asking about information contained in them.

The output you should provide is an answer that uses the inputs as the only source of information, \
while following good costumer service practices.
""".strip()

user_prompt = """
Query:
{query}

Documents:
{documents}
""".strip()

system_prompt = system_prompt.format(expertise_area=expertise_area)
print(system_prompt)

You are a chatbot, expert in the chess club Nau64.

The inputs you will recieve are:
- A list of documents.
- A query asking about information contained in them.

The output you should provide is an answer that uses the inputs as the only source of information, while following good costumer service practices.


In [51]:
def fill_user_prompt(
    query: str,
    documents: list[str],
    # past_conversations: list[str],
    user_prompt=user_prompt,
):
    # # Past conversations
    # past_conversations_string = ""
    # past_conversations_length = len(past_conversations)
    # for idx, user_detail in enumerate(past_conversations):
    #     past_conversations_string += f"- {user_detail}"
    #     if idx < past_conversations_length - 1:
    #         past_conversations_string += "\n"

    # Document
    document_string = ""
    first_document_index = 1
    document_length = len(documents)
    for idx, document in enumerate(documents, start=first_document_index):
        document_string += f"{idx}. {document}"
        if idx < document_length:
            document_string += "\n"


    # Final string
    final_user_prompt = user_prompt.format(
        # past_conversations=past_conversations_string, 
        query=query, 
        documents=document_string
    )
    return final_user_prompt

# Output

In [52]:
formated_prompt = fill_user_prompt(query, retrieved_documents)
print(formated_prompt)

Query:
Qué es nau64?

Documents:
1. 

Skip to content







Menu


Home
Qué es nau64?
Clases de Ajedrez

Clases y Talleres
Clases para Niños
Clases Online


Torneos
Contacto
Nau64 en ChessCom
Uruguay
Área de Alumnos

Plataforma
Cursos y Materiales


 
Search for:






Search










Menu


nau64 en Chess.Com
Club nau64 en Chess.com
Súmate a Chess.Com
 


Twitch
YouTube
Instagram
Facebook
Twitter
TikTok
 







 
chess club
Academia de Ajedrez




Chatea con nosotros    
 










Menu


Home
Qué es nau64?
Clases de Ajedrez

2. ificación olímpica





Nicolás Kulik arrasa en el sabatino de la Academia





Frank y Federico brillan en el Rápido y Blitz Sabatino.





Open 135 el 6 de abril Torneo Pensado (FIDE)





La Batalla Infinita: Karpov vs. Kasparov





Sábados a Todo Ajedrez






lunes, junio 08, 2026














Qué es nau64? 

Un lugar donde 
aprendizaje y entrenamiento; información y formación; o esparcimiento y 
recreación son valorados de igual manera, como distin